In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

## 1. Загружаем разбиение данных

In [ ]:
# зависимости
import torch
import json
import os
import gc
import random
from datetime import datetime
from dataclasses import dataclass, asdict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, TrainerCallback
)
from datasets import load_dataset

random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
path_ = "/content/gdrive/MyDrive/TelegramData/prepared_data"

stack_ = lambda x: os.path.join(path_, x)
train_raw = load_dataset("json", data_files=stack_("train.json"), split="train")
val_raw = load_dataset("json", data_files=stack_("val.json"), split="train")
test_raw = load_dataset("json", data_files=stack_("test.json"), split="train")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def preprocess(ex, tokenizer, max_post_len, max_total_len):
    prompt = ex["input_text"] + "\nКомментарий: "
    target = ex["output_text"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=max_post_len,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids

    # финальное ограничение длины
    input_ids = input_ids[:max_total_len]
    labels = labels[:max_total_len]

    # ручной паддинг
    pad_len = max_total_len - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * (max_total_len - pad_len) + [0] * pad_len
    }

In [ ]:
eval_indices = random.sample(range(len(val_raw)), 100)
eval_posts = [val_raw["input_text"][i] for i in eval_indices]
sample_indices = random.sample(range(100), 10)

# Experiments

In [ ]:
def log_experiment(config, train_log, metrics, path="/content/gdrive/MyDrive/diploma/experiment_log.json"):
    """Дописывает результаты эксперимента в json-файл"""
    entry = {
        "timestamp": datetime.now().isoformat(),
        "config": asdict(config),
        "train_history": train_log,  # из trainer.state.log_history
        "metrics": metrics,           # distinct-n, bert-score и т.д.
    }

    try:
        with open(path, "r") as f:
            log = json.load(f)
    except FileNotFoundError:
        log = []

    log.append(entry)
    with open(path, "w") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class GenerationMetricsCallback(TrainerCallback):
    def __init__(self, eval_posts, sample_indices, tokenizer, max_post_len):
        self.eval_posts = eval_posts
        self.sample_indices = sample_indices
        self.tokenizer = tokenizer
        self.epoch_metrics = []
        self.max_post_len = max_post_len

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        generated = generate_comments(model, self.tokenizer, self.eval_posts, max_post_len=self.max_post_len)
        d2_tok = distinct_n_tokenized(generated, self.tokenizer, 2)
        self_d2 = mean_self_distinct(generated, self.tokenizer, 2)
        comb_d2 = combined_distinct(d2_tok, self_d2)

        metrics = {
            "epoch": state.epoch,
            "distinct_1": distinct_n(generated, 1),
            "distinct_2": distinct_n(generated, 2),
            "distinct_2_tok": d2_tok,
            "self_distinct_2": self_d2,
            "combined_distinct_2": comb_d2,
            "samples": [
                {"post": self.eval_posts[i], "generated": generated[i]}
                for i in self.sample_indices
            ]
        }
        self.epoch_metrics.append(metrics)
        print(f"Epoch {state.epoch}: distinct_1={metrics['distinct_1']:.3f}, distinct_2={metrics['distinct_2']:.3f}", f"d2_tok={metrics['distinct_2_tok']:.3f}, self_d2={metrics['self_distinct_2']:.3f}, comb_d2={metrics['combined_distinct_2']:.3f}")

In [ ]:
def generate_comments(model, tokenizer, posts, max_post_len=256, max_new_tokens=64):
    model.eval()
    generated = []
    for i, post in enumerate(posts):
        torch.manual_seed(42 + i)
        prompt = f"{post}\nКомментарий:"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                          max_length=max_post_len).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                do_sample=True,
                max_new_tokens=max_new_tokens,
                top_p=0.9,
                temperature=0.7,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id
            )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                               skip_special_tokens=True)
        generated.append(text)
    return generated

def distinct_n(texts, n=2):
    all_ngrams = []
    for text in texts:
        tokens = text.split()
        ngrams = list(zip(*[tokens[i:] for i in range(n)]))
        all_ngrams.extend(ngrams)
    if not all_ngrams:
        return 0.0
    return len(set(all_ngrams)) / len(all_ngrams)

def distinct_n_tokenized(texts, tokenizer, n=2):
    "Distinct-n на уровне токенов, а не слов"
    all_ngrams = []
    for text in texts:
        ids = tokenizer.encode(text, add_special_tokens=False)
        ngrams = list(zip(*[ids[i:] for i in range(n)]))
        all_ngrams.extend(ngrams)
    if not all_ngrams:
        return 0.0
    return len(set(all_ngrams)) / len(all_ngrams)


def mean_self_distinct(texts, tokenizer, n=2):
    "Средний distinct-n внутри каждого комментария — детектор шизофазий"
    scores = []
    for text in texts:
        ids = tokenizer.encode(text, add_special_tokens=False)
        ngrams = list(zip(*[ids[i:] for i in range(n)]))
        if len(ngrams) == 0:
            continue
        scores.append(len(set(ngrams)) / len(ngrams))
    if not scores:
        return 0.0
    return sum(scores) / len(scores)


def combined_distinct(global_d, self_d, n=2, beta=1):
    "Произведение глобального distinct на mean self-distinct"
    if beta == 1:
      return global_d * self_d
    else:
      return global_d * self_d ** beta

In [ ]:
model_name1 = "sberbank-ai/rugpt3medium_based_on_gpt2"
model_name2 = "sberbank-ai/rugpt3large_based_on_gpt2"

In [ ]:
@dataclass
class ExperimentConfig:
  experiment_name: str = "baseline"
  max_post_len: int = 128
  max_total_len: int = 192
  num_epochs: int = 3
  lr: float = 5e-5
  gradient_accumulation_steps: int = 1
  weight_decay: float = 0.001
  model_name: str = "sberbank-ai/rugpt3small_based_on_gpt2"
  train_batch: int = 8
  eval_batch: int = 8
  logging_steps: int = 50
  base_dir: str = "/content/gdrive/MyDrive/diploma"
  @property
  def logging_dir(self):
      return f"{self.base_dir}/logs/{self.experiment_name}"

  @property
  def output_dir(self):
      return f"{self.base_dir}/results/{self.experiment_name}"

In [ ]:
def run_experiment(train_raw, val_raw, config: ExperimentConfig):
  cleanup()
  print(f"Running: {asdict(config)}")

  tokenizer = AutoTokenizer.from_pretrained(config.model_name)
  # определяем токен окончания генерации
  if tokenizer.eos_token is None:
    tokenizer.eos_token = "</s>"
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  model = AutoModelForCausalLM.from_pretrained(
      config.model_name,
      # for qwen-only
      torch_dtype=torch.float32,
      )

  preprocess_fn = lambda ex: preprocess(ex, tokenizer, config.max_post_len, config.max_total_len)
  train_data = train_raw.map(preprocess_fn)
  val_data = val_raw.map(preprocess_fn)

  callback = GenerationMetricsCallback(eval_posts, sample_indices, tokenizer, config.max_post_len)

  training_args = TrainingArguments(
    output_dir=config.output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=config.lr,
    per_device_train_batch_size=config.train_batch,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    per_device_eval_batch_size=config.eval_batch,
    num_train_epochs=config.num_epochs,
    weight_decay=config.weight_decay,
    lr_scheduler_type="cosine",
    logging_dir=config.logging_dir,
    logging_steps=config.logging_steps,
    push_to_hub=False,
    # for qwen
    fp16=True,
    bf16=False,
    # for gpt-2
    # fp16=torch.cuda.is_available(),

    report_to=["tensorboard"],
    seed=42,
    data_seed=42,

  )

  # создаем хф-трейнер
  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_data,
      eval_dataset=val_data,
      callbacks=[callback],
  )
  trainer.train()

  log_experiment(config, trainer.state.log_history, callback.epoch_metrics, path=os.path.join(config.base_dir, "experiment_log.json"))

  cleanup()
  return

In [ ]:
configs = [
    # ExperimentConfig(model_name=model_name1, max_post_len=128, max_total_len=192, experiment_name="gpt2_128_192"),
    ExperimentConfig(model_name=model_name1, max_post_len=256, max_total_len=384, experiment_name="gpt2_256_384_medium"),
    # ExperimentConfig(model_name=model_name1, max_post_len=384, max_total_len=512, experiment_name="gpt2_384_512"),
]

In [ ]:
import numpy as np


tokenizer_gpt = AutoTokenizer.from_pretrained(model_name1)
post_lens = [len(ids) for ids in  tokenizer_gpt(list(train_raw["input_text"]))['input_ids']]
comment_lens = [len(ids) for ids in tokenizer_gpt(list(train_raw["output_text"]))['input_ids']]

for name, lens in [("Posts", post_lens), ("Comments", comment_lens)]:
  print(f"{name}: mean={np.mean(lens): .0f}, "
        f"p50={np.percentile(lens, 50):.0f}, "
        f"p90={np.percentile(lens, 90):.0f}, "
        f"p95={np.percentile(lens, 95):.0f}, "
        f"p99={np.percentile(lens, 99):.0f}, "
        f"fit_in_128={np.mean(np.array(lens) <= 128):.1%}",
        f"fit_in_256={np.mean(np.array(lens) <= 256):.1%}",
        f"fit_in_384={np.mean(np.array(lens) <= 384):.1%}")


In [ ]:
tokenizer_qwen = AutoTokenizer.from_pretrained(model_name2)
post_lens = [len(ids) for ids in  tokenizer_qwen(list(train_raw["input_text"]))['input_ids']]
comment_lens = [len(ids) for ids in tokenizer_qwen(list(train_raw["output_text"]))['input_ids']]

for name, lens in [("Posts", post_lens), ("Comments", comment_lens)]:
  print(f"{name}: mean={np.mean(lens): .0f}, "
        f"p50={np.percentile(lens, 50):.0f}, "
        f"p90={np.percentile(lens, 90):.0f}, "
        f"p95={np.percentile(lens, 95):.0f}, "
        f"p99={np.percentile(lens, 99):.0f}, "
        f"fit_in_128={np.mean(np.array(lens) <= 128):.1%}",
        f"fit_in_256={np.mean(np.array(lens) <= 256):.1%}",
        f"fit_in_384={np.mean(np.array(lens) <= 384):.1%}")

In [ ]:
cleanup()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/gdrive/MyDrive/diploma/results/ru
for config in configs:
  run_experiment(train_raw, val_raw, config)

## rugpt3_sber_large

In [ ]:
configs_qwen = [
    # ExperimentConfig(model_name=model_name2, max_post_len=128, max_total_len=192, experiment_name="gpt2_128_192"),
    # ExperimentConfig(model_name=model_name2, gradient_accumulation_steps=4, train_batch=2, eval_batch=2, num_epochs=3, max_post_len=256, max_total_len=384, experiment_name="qwen2_5_256_384", base_dir="/content/gdrive/MyDrive/diploma/Qwen"),
    ExperimentConfig(model_name=model_name2, gradient_accumulation_steps=8, train_batch=1, eval_batch=2, num_epochs=3, max_post_len=256, max_total_len=384, experiment_name="gpt2_256_384_large", base_dir="/content/gdrive/MyDrive/diploma/results"),
]

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/gdrive/MyDrive/diploma/results/

In [ ]:
for config in configs_qwen:
  run_experiment(train_raw, val_raw, config)

In [ ]:
cleanup()

In [ ]:
!find /content -name "experiment_log.json" 2>/dev/null

In [ ]:
transformers.__version__